# Практическая работа № 2
## Верификация и оценка качества образовательных данных: от raw-слоя к clean-слою в Google Colab и BigQuery

**Курс:** «Методы анализа больших данных» · **Раздел 2:** «Получение и подготовка больших данных» · **Лекция 2**

| Поле | Заполните |
|---|---|
| ФИО |  |
| Группа |  |
| Номер варианта |  |
| Дата |  |

### Сквозной кейс

Вы получили выгрузку результатов заданий из LMS. Прежде чем считать средние баллы и строить модели,
нужно ответить на вопрос лекции 2: **можно ли этим данным доверять?**

**raw CSV → профилирование → правила качества → конвейер очистки → clean-слой в pandas и BigQuery → сверка → паспорт данных**

### Что важно

1. Все работают с одним файлом **`practice02_raw.csv`**, вариант задаёт **курс и поток** (срез данных).
2. В выгрузке есть типичные дефекты. **Какие именно — вы находите сами** на шаге профилирования.
3. Строки с проблемными значениями **не удаляются молча**: значение заменяется пропуском, причина записывается во флаг.
4. Ячейки с пометкой **`TODO`** нужно дополнить. Автопроверки после них подскажут, если что-то не так.
5. Выполняйте ячейки сверху вниз. После правки `TODO` перезапустите ячейки ниже.

## Теоретический минимум

### Слои данных (лекция 2, раздел 1)

| Слой | Где у нас | Правило |
|---|---|---|
| **raw** | `practice02_raw.csv`, таблица `p02_raw` | хранится «как есть», все поля — текст, не правится вручную |
| **clean** | `clean` в pandas, представление `p02_clean` | каждое изменение — код; причина фиксируется флагом |
| **analytic** | `p02_analytic` (по желанию) | агрегаты под конкретную задачу |

### Поля файла

| Поле | Смысл |
|---|---|
| `event_id` | идентификатор записи в выгрузке |
| `code_module`, `code_presentation` | курс и поток (ваш вариант) |
| `id_assessment`, `assessment_type` | задание и его тип: `TMA` — работа с проверкой тьютором, `CMA` — компьютерный тест, `Exam` |
| `assessment_deadline`, `weight` | срок сдачи (день от начала курса) и вес задания |
| `id_student`, `date_submitted` | обезличенный студент и день сдачи |
| `is_banked` | 1 — результат перезачтён из прошлого потока |
| `score` | балл по шкале 0–100 |
| `final_result` | итог курса: Pass, Distinction, Fail, Withdrawn |

### Показатели качества (лекция 2, раздел 10)

| Показатель | Как считаем в этой работе |
|---|---|
| **C** — полнота | доля строк, где `score` известен (не пусто и не служебный код) |
| **V** — валидность | доля значений в шкале 0–100 **среди заполненных** |
| **U** — уникальность | число уникальных ключей `(id_assessment, id_student)` / число строк |
| **S** — согласованность | доля строк, где тип задания записан по словарю и нет противоречия «Withdrawn + Exam» |

$$DQ = 0{,}35C + 0{,}30V + 0{,}20U + 0{,}15S$$

Веса — учебная договорённость, а не стандарт. Высокое DQ может скрывать проблему в одном измерении.

### Два вида дублей (лекция 2, раздел 8)

- **точный дубль** — строка повторяется целиком;
- **логический дубль** — та же сдача (то же задание, тот же студент) пришла повторно с другим `event_id`.

## Шаг 0. Ваш вариант

| № | Модуль | Поток | № | Модуль | Поток |
|---:|---|---|---:|---|---|
| 1 | `AAA` | `2013J` | 11 | `EEE` | `2013J` |
| 2 | `BBB` | `2013B` | 12 | `EEE` | `2014B` |
| 3 | `BBB` | `2014B` | 13 | `EEE` | `2014J` |
| 4 | `BBB` | `2014J` | 14 | `FFF` | `2013B` |
| 5 | `CCC` | `2014B` | 15 | `FFF` | `2013J` |
| 6 | `CCC` | `2014J` | 16 | `FFF` | `2014B` |
| 7 | `DDD` | `2013B` | 17 | `FFF` | `2014J` |
| 8 | `DDD` | `2013J` | 18 | `GGG` | `2013J` |
| 9 | `DDD` | `2014B` | 19 | `GGG` | `2014B` |
| 10 | `DDD` | `2014J` | 20 | `GGG` | `2014J` |

Укажите номер варианта — курс и поток подставятся автоматически.

In [ ]:
# TODO-0. Номер вашего варианта (1–20)
VARIANT = None

VARIANTS = {
    1: ("AAA", "2013J"),
    2: ("BBB", "2013B"),
    3: ("BBB", "2014B"),
    4: ("BBB", "2014J"),
    5: ("CCC", "2014B"),
    6: ("CCC", "2014J"),
    7: ("DDD", "2013B"),
    8: ("DDD", "2013J"),
    9: ("DDD", "2014B"),
    10: ("DDD", "2014J"),
    11: ("EEE", "2013J"),
    12: ("EEE", "2014B"),
    13: ("EEE", "2014J"),
    14: ("FFF", "2013B"),
    15: ("FFF", "2013J"),
    16: ("FFF", "2014B"),
    17: ("FFF", "2014J"),
    18: ("GGG", "2013J"),
    19: ("GGG", "2014B"),
    20: ("GGG", "2014J"),
}

assert VARIANT in VARIANTS, "Укажите номер варианта от 1 до 20 в переменной VARIANT"
MODULE, PRESENTATION = VARIANTS[VARIANT]
print(f"Вариант {VARIANT}: курс {MODULE}, поток {PRESENTATION}")

## Шаг 1. Библиотеки и загрузка raw-файла

Выполните ячейки. Colab предложит выбрать файл — выберите **`practice02_raw.csv`**, выданный преподавателем.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)

In [ ]:
DATA_PATH = Path("practice02_raw.csv")
if not DATA_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Файл не выбран.")
    DATA_PATH = Path(next(iter(uploaded)))
print("Файл:", DATA_PATH)

In [ ]:
# Весь файл читаем как ТЕКСТ (dtype=str): raw-слой хранится «как есть»,
# служебные коды вроде '?' не должны молча превратиться в NaN или число.
raw_all = pd.read_csv(DATA_PATH, dtype=str)

n_pres = raw_all[["code_module", "code_presentation"]].drop_duplicates().shape[0]
print(f"Весь файл: строк {len(raw_all):,}, столбцов {raw_all.shape[1]}, "
      f"уникальных event_id {raw_all['event_id'].nunique():,}, презентаций курсов {n_pres}")

raw = raw_all[(raw_all["code_module"] == MODULE)
              & (raw_all["code_presentation"] == PRESENTATION)].copy()
print(f"Срез {MODULE} {PRESENTATION}: строк {len(raw):,}")
assert len(raw) > 0, "Срез пуст: проверьте MODULE и PRESENTATION"
raw.head()

### Контрольные значения

Для всего файла у всех студентов должно получиться:

```text
строк:               176172
столбцов:            12
уникальных event_id: 174781
презентаций курсов:  22
```

Если значения другие — **не продолжайте**, проверьте файл. Число строк вашего среза сверьте с таблицей вариантов в `README`.

> Почему `dtype=str`? По умолчанию pandas превратит `'999'` и `'-1'` в обычные числа, и они молча войдут в средний балл.
> Raw-слой читается «как есть», а типы приводятся позже — осознанно.

## Шаг 2. Профилирование raw-слоя: что не так с данными?

Готовая ячейка показывает шесть «рентгеновских снимков» вашего среза. Внимательно прочитайте вывод.

In [ ]:
KEY_CANDIDATE = ["id_assessment", "id_student"]   # «кто сдал какое задание»

print("1) Типы столбцов raw-слоя:", raw.dtypes.astype(str).unique().tolist())
print("   Пустых ячеек (NaN):", int(raw.isna().sum().sum()))

num = pd.to_numeric(raw["score"], errors="coerce")
print("\n2) Нечисловые значения score:")
print(raw.loc[num.isna(), "score"].value_counts(dropna=False).to_string())

print("\n3) Числовые значения score вне шкалы 0–100 (значение → сколько раз):")
print(raw.loc[num.notna() & ~num.between(0, 100), "score"].value_counts().to_string())

print("\n4) Написание assessment_type:")
print(raw["assessment_type"].value_counts().to_string())

print("\n5) assessment_deadline = '?':", int((raw["assessment_deadline"] == "?").sum()),
      "| из них по типам:", raw.loc[raw["assessment_deadline"] == "?", "assessment_type"]
      .str.upper().value_counts().to_dict())

n_exact = int(raw.duplicated().sum())
n_event_rep = int(raw["event_id"].duplicated().sum())
n_key_rep = int(raw.drop_duplicates().duplicated(subset=KEY_CANDIDATE).sum())
print("\n6) Полностью одинаковых строк (точные дубли):", n_exact)
print("   Повторов event_id:", n_event_rep)
print("   Повторов пары (id_assessment, id_student) после удаления точных дублей:", n_key_rep)

### 2б. Значения `score` вне шкалы во всём файле

Чтобы отличить служебный код от ошибки ввода, посмотрим на частоты по всем 22 потокам.

In [ ]:
num_all = pd.to_numeric(raw_all["score"], errors="coerce")
out_all = raw_all.loc[num_all.notna() & ~num_all.between(0, 100), "score"].value_counts()
print("Значения вне шкалы 0–100 во всём файле (значение → сколько раз):")
print(out_all.to_string())

### Ваши выводы по профилированию

Ответьте кратко (по 1–2 предложения):

1. Сколько пустых ячеек (`NaN`) в срезе? Значит ли это, что пропусков нет?
2. Какие значения `score` означают **«нет данных»** (служебные коды)? Почему вы так решили?
   *Подсказка:* одно и то же невозможное значение, которое повторяется сотни раз, — это код;
   разные значения чуть выше шкалы, по несколько раз каждое, — ошибки ввода.
3. Какие варианты написания `assessment_type` встречаются?
4. Сколько точных и сколько логических дублей? Чем они отличаются?

> **Ваши выводы:**
> 1. ...
> 2. ...
> 3. ...
> 4. ...

## Шаг 3. Правила качества (TODO)

Перенесите выводы профилирования в правила — это и есть **контракт данных** (лекция 2, раздел 6).

In [ ]:
# TODO-1. Служебные коды «нет данных» в поле score (строки!). Один код уже указан — дополните множество.
MISSING_CODES = {"?"}

# Ключ уникальности события «студент сдал задание»
KEY = ["id_assessment", "id_student"]

# TODO-2. Словарь нормализации типа задания.
# Ключ — значение после .strip().upper(), значение — эталонное написание из словаря {TMA, CMA, Exam}.
TYPE_MAP = {"TMA": "TMA"}

In [ ]:
# Автопроверка правил (по всему файлу)
msgs = []
for value, cnt in out_all.items():
    if cnt >= 50 and value not in MISSING_CODES:
        msgs.append(f"значение {value!r} встречается {cnt} раз — похоже на служебный код; добавьте его в MISSING_CODES")
    if cnt < 50 and value in MISSING_CODES:
        msgs.append(f"значение {value!r} встречается всего {cnt} раз — это ошибка ввода, а не код; уберите из MISSING_CODES")
non_numeric = set(raw_all.loc[num_all.isna(), "score"].dropna())
if not non_numeric <= MISSING_CODES:
    msgs.append(f"нечисловые значения {non_numeric - MISSING_CODES} не учтены в MISSING_CODES")
spellings = set(raw_all["assessment_type"].str.strip().str.upper())
if not spellings <= set(TYPE_MAP):
    msgs.append(f"в TYPE_MAP нет ключей для {spellings - set(TYPE_MAP)}")
if not set(TYPE_MAP.values()) <= {"TMA", "CMA", "Exam"}:
    msgs.append("значения TYPE_MAP должны быть из словаря {TMA, CMA, Exam} — с учётом регистра")
print("Правила заданы верно." if not msgs else "Исправьте:\n- " + "\n- ".join(msgs))

## Шаг 4. Показатели качества raw-слоя

Выполните ячейку: функции считают C, V, U, S и DQ по определениям из теоретического минимума.

In [ ]:
WEIGHTS = {"C": 0.35, "V": 0.30, "U": 0.20, "S": 0.15}
DICTIONARY = {"TMA", "CMA", "Exam"}                 # допустимые значения assessment_type


def dq(parts):
    """Интегральная учебная метрика качества DQ (лекция 2, раздел 10)."""
    parts["DQ"] = sum(WEIGHTS[k] * parts[k] for k in WEIGHTS)
    return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in parts.items()}


def quality_raw(df):
    """Показатели качества сырого среза (все поля — строки)."""
    n = len(df)
    missing = df["score"].isna() | df["score"].isin(MISSING_CODES)
    num = pd.to_numeric(df["score"], errors="coerce")
    valid = ~missing & num.between(0, 100)
    withdrawn_exam = (df["final_result"] == "Withdrawn") & \
                     (df["assessment_type"].str.strip().str.upper() == "EXAM")
    return dq({
        "N": n,
        "C": 1 - missing.sum() / n,                          # полнота score
        "V": valid.sum() / (~missing).sum(),                  # валидность среди заполненных
        "U": df[KEY].drop_duplicates().shape[0] / n,          # уникальность ключа
        "S": (df["assessment_type"].isin(DICTIONARY) & ~withdrawn_exam).mean(),  # согласованность
    })


def quality_clean(df):
    """Те же показатели для очищенного среза."""
    n = len(df)
    return dq({
        "N": n,
        "C": df["score"].notna().mean(),
        "V": df["score"].between(0, 100).sum() / df["score"].notna().sum(),
        "U": df[KEY].drop_duplicates().shape[0] / n,
        "S": (~df["withdrawn_exam"]).mean(),
    })


q_raw = quality_raw(raw)
pd.DataFrame([q_raw], index=["raw"])

## Шаг 5. Конвейер очистки raw → clean с журналом строк

Порядок шагов соответствует лекции 2 (раздел 12): **сначала дубли и типы, потом коды и диапазоны,
затем категории и признаки**. Журнал `lineage_df` — основа паспорта данных.

In [ ]:
lineage = []


def log(step, df, note=""):
    lineage.append({"шаг": step, "строк": len(df), "комментарий": note})


log("0. raw-срез варианта", raw)

# 1. Точные дубли: строка повторяется целиком (повторная выгрузка)
step1 = raw.drop_duplicates()
log("1. удалены точные дубли", step1, "повторная выгрузка тех же строк")

# 2. Типы: числовые поля из текста; '?' в сроке сдачи → NaN
step2 = step1.copy()
for col in ["id_assessment", "id_student", "date_submitted", "is_banked"]:
    step2[col] = pd.to_numeric(step2[col], errors="raise").astype("int64")
step2["weight"] = pd.to_numeric(step2["weight"])
step2["assessment_deadline"] = pd.to_numeric(step2["assessment_deadline"].replace("?", np.nan))
log("2. приведены типы", step2, "'?' в assessment_deadline → NaN")

# 3. Логические дубли: одна и та же пара (задание, студент) с разными event_id
step3 = step2.sort_values("event_id").drop_duplicates(subset=KEY, keep="first")
log("3. удалены логические дубли", step3, "оставлена первая доставка (меньший event_id)")

# 4. Служебные коды и доменное правило 0 ≤ score ≤ 100 — строки НЕ удаляем, ставим флаг
step4 = step3.copy()
step4["score_raw"] = step4["score"]
is_code = step4["score_raw"].isna() | step4["score_raw"].isin(MISSING_CODES)
num = pd.to_numeric(step4["score_raw"].where(~is_code), errors="coerce")
in_range = num.between(0, 100)
step4["score_flag"] = np.select([is_code, ~in_range], ["missing_code", "out_of_range"], default="ok")
step4["score"] = num.where(in_range)
log("4. коды пропуска и диапазон score", step4, "значение → NaN + флаг score_flag")

# 5. Нормализация категорий: пробелы, регистр, единый словарь
step5 = step4.copy()
step5["assessment_type"] = step5["assessment_type"].str.strip().str.upper().map(TYPE_MAP)
log("5. нормализованы категории", step5, "tma / Tma → TMA и т. д.")

# 6. Признаки и флаг согласованности
clean = step5.copy()
clean["days_late"] = clean["date_submitted"] - clean["assessment_deadline"]
clean["withdrawn_exam"] = (clean["final_result"] == "Withdrawn") & (clean["assessment_type"] == "Exam")
clean = clean.reset_index(drop=True)
log("6. признаки days_late, withdrawn_exam", clean, "строки не меняются")

lineage_df = pd.DataFrame(lineage)
lineage_df["изменение"] = lineage_df["строк"].diff().fillna(0).astype(int)
lineage_df

In [ ]:
problems = []
if clean.duplicated(subset=KEY).any():
    problems.append("остались повторы ключа (id_assessment, id_student)")
if not clean["score"].dropna().between(0, 100).all():
    problems.append("в score остались значения вне шкалы 0–100")
if clean["assessment_type"].isna().any() or not set(clean["assessment_type"]) <= DICTIONARY:
    problems.append("assessment_type содержит значения вне словаря {TMA, CMA, Exam} — проверьте TYPE_MAP")
if ((clean["score_raw"] == "?") & (clean["score_flag"] != "missing_code")).any():
    problems.append("'?' не распознан как код пропуска — проверьте MISSING_CODES")
if not (clean["score_flag"].eq("ok") == clean["score"].notna()).all():
    problems.append("флаг score_flag не согласован со значением score")
freq = raw_all["score"].value_counts()
frequent_oor = [v for v in clean.loc[clean["score_flag"] == "out_of_range", "score_raw"].unique()
                if freq.get(v, 0) >= 50]
if frequent_oor:
    problems.append(f"значения {frequent_oor} помечены out_of_range, но очень часто встречаются в файле — "
                    "это служебные коды, добавьте их в MISSING_CODES")
removed = (len(raw) - len(step1)) + (len(step2) - len(step3))
if len(clean) != len(raw) - removed:
    problems.append("число строк не сходится с журналом удалений")

print("Все автопроверки пройдены." if not problems else "НАЙДЕНЫ ПРОБЛЕМЫ:\n- " + "\n- ".join(problems))
print("\nФлаги score после очистки:")
print(clean["score_flag"].value_counts().to_string())
print("\nИсходные значения score_raw у флага missing_code:")
print(clean.loc[clean["score_flag"] == "missing_code", "score_raw"].value_counts().to_string())

> **Вопрос.** Почему строки с кодами пропуска и значениями вне шкалы **не удаляются**, а получают флаг?
> Что потерял бы аналитик, если бы просто удалил их?
>
> **Ваш ответ:** ...

## Шаг 6. Качество до и после очистки

In [ ]:
q_clean = quality_clean(clean)
quality = pd.DataFrame([q_raw, q_clean], index=["raw", "clean"])
quality.loc["Δ"] = quality.loc["clean"] - quality.loc["raw"]
display(quality)

dims = ["C", "V", "U", "S", "DQ"]
x = np.arange(len(dims))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, [q_raw[d] for d in dims], width=0.38, label="raw", color="#E45756")
ax.bar(x + 0.2, [q_clean[d] for d in dims], width=0.38, label="clean", color="#4C78A8")
ax.set_xticks(x, dims)
ax.set_ylim(0.97, 1.001)                     # ось обрезана намеренно — подпись ниже
ax.set_ylabel("Значение показателя (ось от 0,97)")
ax.set_title(f"Качество данных до и после очистки: {MODULE} {PRESENTATION}")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

> **Вопрос.** Какие показатели выросли, а какой мог **снизиться**? Объясните, почему после очистки
> полнота C может стать хуже. Обратите внимание на ось графика: с какого значения она начинается и почему.
>
> **Ваш ответ:** ...

## Шаг 7. Выбросы и своевременность

Признак `days_late` — на сколько дней сдача отличается от срока (меньше 0 — раньше срока).
Правило IQR (лекция 2, раздел 7) применяем к работам TMA без перезачтённых результатов.

In [ ]:
tma = clean[(clean["assessment_type"] == "TMA") & (clean["is_banked"] == 0)]["days_late"].dropna()
q1, q3 = tma.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
is_out = (tma < lower) | (tma > upper)

print(f"TMA без перезачтённых (is_banked = 0): {len(tma):,} сдач")
print(f"Q1 = {q1:g}, Q3 = {q3:g}, IQR = {iqr:g}, границы [{lower:g}; {upper:g}]")
print(f"Кандидатов в выбросы: {is_out.sum():,} ({is_out.mean():.1%})")
print(f"  раньше нижней границы: {(tma < lower).sum():,}; позже верхней: {(tma > upper).sum():,}")
print(f"Медиана {tma.median():g} дн., минимум {tma.min():g}, максимум {tma.max():g}")

print("\nПерезачтённые результаты (is_banked) и даты до начала курса:")
display(clean.groupby("is_banked")["date_submitted"]
        .agg(n_rows="count", min_day="min", max_day="max",
             n_negative_day=lambda s: int((s < 0).sum())))

print("Доля сдач позже срока по типам заданий (days_late > 0):")
display(clean.groupby("assessment_type")["days_late"]
        .agg(n_with_deadline="count", share_late=lambda s: round((s > 0).mean(), 3)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(tma.clip(-30, 60), bins=90, color="#4C78A8")
ax.axvline(lower, color="#E45756", linestyle="--", label=f"границы IQR: {lower:g} и {upper:g}")
ax.axvline(upper, color="#E45756", linestyle="--")
ax.set_title(f"Отклонение сдачи TMA от срока, {MODULE} {PRESENTATION} (значения обрезаны до −30…60)")
ax.set_xlabel("days_late = date_submitted − assessment_deadline, дней")
ax.set_ylabel("Число сдач")
ax.legend()
plt.tight_layout()
plt.show()

> **Вопросы.**
> 1. Какая доля сдач помечена как кандидаты в выбросы? Можно ли их удалить? Почему?
> 2. Какие даты у перезачтённых результатов (`is_banked = 1`)? Почему их исключают из анализа своевременности?
> 3. Сравните долю «поздних» сдач для TMA и CMA. Правдоподобно ли, что почти все студенты опаздывают с CMA?
>    Что это говорит о смысле поля `assessment_deadline` для тестов?
> 4. Если в вашем варианте есть экзамены (`Exam`): известен ли их срок? Если `days_late` у экзаменов пустой — почему?
>
> **Ваши ответы:** ...

## Шаг 8. Структура пропусков: MCAR или нет?

Лекция 2, раздел 5: если пропуск связан с итогом обучения, удаление таких строк исказит выборку.

In [ ]:
def missing_table(df):
    t = (df.assign(q=df["score_raw"].eq("?"), codes=df["score_raw"].isin(["999", "-1"]))
           .groupby("final_result")
           .agg(n_rows=("event_id", "size"), n_question_mark=("q", "sum"), n_other_codes=("codes", "sum")))
    t["pct_question_mark"] = (100 * t["n_question_mark"] / t["n_rows"]).round(3)
    t["pct_other_codes"] = (100 * t["n_other_codes"] / t["n_rows"]).round(3)
    return t.sort_values("pct_question_mark", ascending=False)


# Весь файл (все 22 презентации): удаляем дубли так же, как в конвейере
all_dedup = (raw_all.drop_duplicates()
             .sort_values("event_id")
             .drop_duplicates(subset=KEY, keep="first")
             .rename(columns={"score": "score_raw"}))
print("Весь файл — доли кодов пропуска по итоговому результату, %:")
display(missing_table(all_dedup))

print(f"Ваш срез {MODULE} {PRESENTATION}:")
display(missing_table(clean))

> **Вопросы.**
> 1. Сравните долю кода `'?'` у групп Withdrawn и Pass во всём файле. Похоже ли это на MCAR?
> 2. Зависит ли доля кодов `999`/`-1` от итога обучения?
> 3. Что произойдёт с выборкой, если удалить все строки с `'?'`?
> 4. Почему выводы по вашему срезу нужно делать осторожно?
>
> **Ваши ответы:** ...

## Шаг 9. BigQuery: raw-слой, серия SQL-запросов и clean-представление

### Главное правило

**Python выполняется в Colab, SQL — в BigQuery.** В редактор BigQuery нельзя вставлять код из ячеек Colab
(`PROJECT_ID = …`, `SQL = {…}`, `run(...)`): BigQuery понимает только SQL. Если хотите выполнить запрос в консоли —
копируйте **напечатанный** ячейкой текст, начиная с `SELECT` или `CREATE`.

### Как устроен шаг

* Проект, датасет и имена таблиц задаются **один раз** — в ячейке 9.1 (TODO-3). В текстах запросов вместо имён
  стоят метки `{RAW}`, `{CLEAN}`, `{VARIANT}`: функция `run()` сама подставит ваш проект, датасет и вариант.
* Каждый запрос — **отдельная небольшая ячейка**. Она печатает SQL, выполняет его в BigQuery прямо из Colab
  и показывает результат таблицей и **графиком**.
* Под каждым графиком — поле **«Что вы видите?»**: 1–2 предложения своими словами.
* В трёх запросах есть пропуски `___` (**TODO-4, TODO-5, TODO-6**). После каждого — автопроверка: она сравнивает
  ответ BigQuery с вашим конвейером pandas из шага 5.

| Запрос | Имя | Что узнаём | График |
|---:|---|---|---|
| 1 | `q0_control` | загрузилась ли таблица целиком | — |
| 2 | `q0b_presentations` | сколько строк в каждом курсе и потоке | столбцы по курсам |
| 3 | `q1_profile` (**TODO-4**) | какие значения стоят в `score` | столбцы по видам значений |
| 4 | `q1b_types` | как записан тип задания | столбцы по написаниям |
| 5 | `q2_duplicates`, `q2b_dup_examples` | сколько точных и логических дублей | столбцы дублей |
| 6 | `q3_raw_quality` | показатели качества raw-слоя | столбцы C, V, U, S, DQ |
| 7 | `q4a` … `q4_create_clean` (**TODO-5, TODO-6**), `q4e_lineage` | clean-слой: 4 представления | журнал строк |
| 8 | `q5_clean_quality` | качество после очистки | raw против clean |
| 9 | `q5b_flags` | какие баллы помечены флагами | столбцы по флагам |
| 10 | `q6a_days_late`, `q6_iqr`, `q6b_banked` | опоздания и выбросы | гистограмма с границами IQR |
| 11 | `q7_missing_structure` | у кого чаще пропуски балла | столбцы по итогам курса |
| 12 | `q8_check` | сверка с pandas (шаг 10) | средний балл по заданиям |

### Подготовка в BigQuery Studio (один раз)

1. Откройте `https://console.cloud.google.com/bigquery` — вверху должна быть метка **Sandbox**.
2. Скопируйте **Project ID** (строчные буквы, цифры, дефисы — например `quality-management-analysis`).
   Это не отображаемое имя проекта.
3. Датасет: используйте тот, который назвал преподаватель для вашей группы (например `learning_analytics_pr_02`),
   или создайте: **Explorer → ваш проект → ⋮ → Create dataset**, `Dataset ID` = имя датасета,
   `Location type = Multi-region`, `Location = EU`.

> **Sandbox:** `SELECT` и `CREATE VIEW` работают; `INSERT/UPDATE/DELETE` — нет, и они не нужны:
> clean-слой строится как **представления (view)** поверх неизменного raw-слоя.

### 9.1. Настройки (TODO-3)

Впишите **Project ID** и **имя датасета** своей группы и выполните ячейку. Больше нигде в блокноте имена
проекта и датасета не встречаются: если преподаватель дал другие имена таблиц — меняйте их только здесь, в `TABLES`.

In [ ]:
# ==================== 9.1. НАСТРОЙКИ BIGQUERY (меняются только здесь) ====================
# TODO-3. Ваш Project ID и датасет группы
PROJECT_ID = "your-project-id"            # ← ID проекта из консоли BigQuery (не отображаемое имя!)
DATASET    = "learning_analytics_pr_02"   # ← датасет вашей группы (как назвал преподаватель)
TABLES = {                                # имена таблиц и представлений
    "RAW":      "p02_raw",          # raw-слой: CSV «как есть», все поля STRING
    "S1":       "p02_s1_distinct",  # clean, шаг 1: без точных дублей
    "S2":       "p02_s2_typed",     # clean, шаг 2: типы и служебные коды
    "S3":       "p02_s3_dedup",     # clean, шаг 3: без логических дублей
    "CLEAN":    "p02_clean",        # clean-слой: итоговое представление
    "ANALYTIC": "p02_analytic",     # ★ аналитический слой (шаг 11)
}
ENGINE = "auto"   # в Colab запросы выполняются в BigQuery; менять не нужно
# Вариант (курс и поток) уже задан на шаге 0: MODULE, PRESENTATION
assert PROJECT_ID != "your-project-id", "TODO-3: впишите свой Project ID"
# ===========================================================================================

In [ ]:
# 9.2. Служебные функции: подстановка имён, выполнение запроса, графики. Эту ячейку не меняйте.
import re
import matplotlib.pyplot as plt

assert re.fullmatch(r"[a-z][a-z0-9-]{4,28}[a-z0-9]", PROJECT_ID), \
    "PROJECT_ID: строчные латинские буквы, цифры и дефис (это ID проекта, а не его имя)"
assert re.fullmatch(r"[A-Za-z0-9_]+", DATASET), "DATASET: только латиница, цифры и подчёркивание"

if ENGINE == "auto":
    try:
        import google.colab  # noqa: F401
        ENGINE = "bigquery"
    except ImportError:
        ENGINE = "duckdb"


def fq(key):
    """Полное имя объекта BigQuery: `проект.датасет.таблица`."""
    return f"`{PROJECT_ID}.{DATASET}.{TABLES[key]}`"


def fill(template):
    """Вместо меток {RAW}, {CLEAN}, …, {VARIANT} подставляет ваши таблицы и ваш вариант."""
    names = {k: fq(k) for k in TABLES}
    return template.format(**names, VARIANT=f"code_module = '{MODULE}' AND code_presentation = '{PRESENTATION}'")


def sql(name):
    """Готовый запрос из словаря SQL."""
    return fill(SQL[name])


def to_duckdb(q):
    """Тот же SQL в диалекте DuckDB (для локального эталона)."""
    q = re.sub(r"`[\w-]+\.\w+\.(\w+)`", r"\1", q)
    for a, b in [("SAFE_CAST", "TRY_CAST"), ("INT64", "BIGINT"), ("FLOAT64", "DOUBLE"),
                 ("COUNTIF(", "count_if("), ("PERCENTILE_CONT(", "quantile_cont("),
                 ("SAFE_DIVIDE(", "safe_divide("), ("AS STRING", "AS VARCHAR")]:
        q = q.replace(a, b)
    return q


if ENGINE == "bigquery":
    from google.colab import auth
    from google.cloud import bigquery
    auth.authenticate_user()
    client = bigquery.Client(project=PROJECT_ID)
else:
    import duckdb
    con = duckdb.connect()
    con.execute("CREATE MACRO safe_divide(a, b) AS CASE WHEN b = 0 THEN NULL ELSE a / b END")
    con.execute(f"CREATE OR REPLACE TABLE {TABLES['RAW']} AS "
                f"SELECT * FROM read_csv('{DATA_PATH}', all_varchar=true, header=true)")


def execute(text, title="запрос", show_sql=True):
    """Печатает SQL, выполняет его и возвращает результат как таблицу pandas."""
    if "___" in text:
        raise ValueError(f"В запросе «{title}» остался пропуск ___ — заполните его (см. TODO в ячейке).")
    if show_sql:
        print(f"-- ===== {title} =====\n{text}\n")
    is_create = text.lstrip().upper().startswith("CREATE")
    if ENGINE == "bigquery":
        job = client.query(text)
        job.result()
        if is_create:
            print("Представление создано:", re.search(r"VIEW (\S+)", text).group(1)); return None
        df = job.to_dataframe()
        for c in df.columns:                      # nullable-типы BigQuery → обычные числа pandas
            if str(df[c].dtype) in ("Int64", "Float64", "boolean"):
                df[c] = df[c].astype("float64") if df[c].isna().any() else df[c].astype(str(df[c].dtype).lower())
        return df
    res = con.execute(to_duckdb(text))
    if is_create:
        print("Представление создано:", re.search(r"VIEW (\S+)", text).group(1)); return None
    return res.df()


def run(name, show_sql=True):
    """Запрос из словаря SQL по имени: run("q0_control")."""
    return execute(sql(name), name, show_sql)


def run_custom(text, show_sql=True):
    """Ваш собственный запрос с метками {RAW}, {CLEAN}, {VARIANT}: run_custom(\"\"\"SELECT …\"\"\")."""
    return execute(fill(text), "собственный запрос", show_sql)


BLUE, ORANGE, GREY, GREEN = "#2a78d6", "#eb6834", "#9aa3ad", "#1baf7a"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})


def bars(labels, values, title, ylabel="", colors=None, horizontal=False, fmt="{:,.0f}", figsize=(9, 3.6)):
    """Столбчатая диаграмма с подписями значений."""
    labels = [str(x) for x in labels]
    values = [float(v) for v in values]
    colors = colors or [BLUE] * len(values)
    fig, ax = plt.subplots(figsize=figsize)
    if horizontal:
        ax.barh(labels, values, color=colors); ax.invert_yaxis(); ax.set_xlabel(ylabel)
        for i, v in enumerate(values):
            ax.text(v, i, " " + fmt.format(v).replace(",", " "), va="center", fontsize=8)
    else:
        ax.bar(labels, values, color=colors); ax.set_ylabel(ylabel)
        for i, v in enumerate(values):
            ax.text(i, v, fmt.format(v).replace(",", " "), ha="center", va="bottom", fontsize=8)
    ax.set_title(title)
    plt.tight_layout(); plt.show()


print(f"Режим выполнения: {ENGINE}")
for k in TABLES:
    print(f"  {k:<9} {fq(k)}")
print(f"Вариант: {MODULE} {PRESENTATION}")

In [ ]:
# 9.3. Тексты запросов. Эту ячейку не меняйте — просто выполните.
# Метки {RAW}, {S1}, {S2}, {S3}, {CLEAN}, {ANALYTIC} — таблицы, {VARIANT} — ваш вариант.
# Три запроса (q1_profile, q4c_s3_dedup, q4_create_clean) вы допишете сами — в ячейках запросов 3 и 7.
SQL = {}

SQL["q0_control"] = """SELECT
  COUNT(*)                                                    AS rows_count,
  COUNT(DISTINCT event_id)                                    AS events,
  COUNT(DISTINCT id_student)                                  AS students,
  COUNT(DISTINCT CONCAT(code_module, '-', code_presentation)) AS presentations,
  COUNTIF(event_id = 'event_id')                              AS header_as_data
FROM {RAW};"""

SQL["q0b_presentations"] = """SELECT
  CONCAT(code_module, ' ', code_presentation) AS presentation,
  COUNT(*)                                    AS n_rows
FROM {RAW}
GROUP BY presentation
ORDER BY presentation;"""

SQL["q1b_types"] = """SELECT
  assessment_type,
  COUNT(*) AS n_rows
FROM {RAW}
WHERE {VARIANT}
GROUP BY assessment_type
ORDER BY n_rows DESC;"""

SQL["q2_duplicates"] = """SELECT
  (SELECT COUNT(*) FROM {RAW} WHERE {VARIANT})                          AS n_rows,
  (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {RAW} WHERE {VARIANT})) AS n_distinct_rows,
  (SELECT COUNT(DISTINCT CONCAT(id_assessment, '|', id_student))
     FROM {RAW} WHERE {VARIANT})                                         AS n_keys;"""

SQL["q2b_dup_examples"] = """SELECT
  id_assessment,
  id_student,
  COUNT(*)                                     AS n_copies,
  STRING_AGG(event_id, ', ' ORDER BY event_id) AS event_ids,
  STRING_AGG(score, ', ' ORDER BY event_id)    AS scores
FROM (SELECT DISTINCT * FROM {RAW} WHERE {VARIANT})
GROUP BY id_assessment, id_student
HAVING COUNT(*) > 1
ORDER BY id_assessment, id_student
LIMIT 5;"""

SQL["q3_raw_quality"] = """WITH m AS (
  SELECT
    COUNT(*)                                                     AS n,
    COUNTIF(score IS NULL OR score IN ('?', '999', '-1'))        AS n_missing,
    COUNTIF(SAFE_CAST(score AS INT64) BETWEEN 0 AND 100)         AS n_valid,
    COUNT(DISTINCT CONCAT(id_assessment, '|', id_student))       AS n_keys,
    COUNTIF(assessment_type IN ('TMA', 'CMA', 'Exam')
            AND NOT (final_result = 'Withdrawn'
                     AND UPPER(TRIM(assessment_type)) = 'EXAM')) AS n_consistent
  FROM {RAW}
  WHERE {VARIANT}
)
SELECT
  n,
  ROUND(1 - n_missing / n, 4)         AS C,
  ROUND(n_valid / (n - n_missing), 4) AS V,
  ROUND(n_keys / n, 4)                AS U,
  ROUND(n_consistent / n, 4)          AS S,
  ROUND(0.35 * (1 - n_missing / n) + 0.30 * n_valid / (n - n_missing)
      + 0.20 * n_keys / n + 0.15 * n_consistent / n, 4) AS DQ
FROM m;"""

SQL["q4a_s1_distinct"] = """CREATE OR REPLACE VIEW {S1} AS
SELECT DISTINCT *          -- шаг 1: убрать точные дубли (строка повторена целиком)
FROM {RAW};"""

SQL["q4b_s2_typed"] = """CREATE OR REPLACE VIEW {S2} AS
SELECT                     -- шаг 2: привести типы, служебные коды балла → NULL
  event_id,
  code_module,
  code_presentation,
  SAFE_CAST(id_assessment AS INT64)                    AS id_assessment,
  assessment_type                                      AS assessment_type_raw,
  SAFE_CAST(NULLIF(assessment_deadline, '?') AS INT64) AS assessment_deadline,
  SAFE_CAST(weight AS FLOAT64)                         AS weight,
  SAFE_CAST(id_student AS INT64)                       AS id_student,
  SAFE_CAST(date_submitted AS INT64)                   AS date_submitted,
  SAFE_CAST(is_banked AS INT64)                        AS is_banked,
  score                                                AS score_raw,
  IF(score IN ('?', '999', '-1'), NULL, SAFE_CAST(score AS INT64)) AS score_int,
  final_result
FROM {S1};"""

SQL["q4e_lineage"] = """SELECT '1. raw' AS step, COUNT(*) AS n_rows FROM {RAW} WHERE {VARIANT}
UNION ALL SELECT '2. без точных дублей', COUNT(*) FROM {S1} WHERE {VARIANT}
UNION ALL SELECT '3. без логических дублей', COUNT(*) FROM {S3} WHERE {VARIANT}
UNION ALL SELECT '4. clean', COUNT(*) FROM {CLEAN} WHERE {VARIANT}
ORDER BY step;"""

SQL["q5_clean_quality"] = """WITH m AS (
  SELECT
    COUNT(*)                                     AS n,
    COUNTIF(score IS NOT NULL)                   AS n_present,
    COUNTIF(score BETWEEN 0 AND 100)             AS n_valid,
    COUNT(DISTINCT CONCAT(CAST(id_assessment AS STRING), '|',
                          CAST(id_student AS STRING)))  AS n_keys,
    COUNTIF(NOT withdrawn_exam)                  AS n_consistent,
    COUNTIF(score_flag = 'missing_code')         AS n_missing_code,
    COUNTIF(score_flag = 'out_of_range')         AS n_out_of_range
  FROM {CLEAN}
  WHERE {VARIANT}
)
SELECT
  n, n_missing_code, n_out_of_range,
  ROUND(n_present / n, 4)       AS C,
  ROUND(n_valid / n_present, 4) AS V,
  ROUND(n_keys / n, 4)          AS U,
  ROUND(n_consistent / n, 4)    AS S,
  ROUND(0.35 * n_present / n + 0.30 * n_valid / n_present
      + 0.20 * n_keys / n + 0.15 * n_consistent / n, 4) AS DQ
FROM m;"""

SQL["q5b_flags"] = """SELECT
  score_flag,
  score_raw,
  COUNT(*) AS n_rows
FROM {CLEAN}
WHERE {VARIANT} AND score_flag != 'ok'
GROUP BY score_flag, score_raw
ORDER BY score_flag, n_rows DESC;"""

SQL["q6a_days_late"] = """SELECT
  days_late,
  COUNT(*) AS n_rows
FROM {CLEAN}
WHERE {VARIANT}
  AND assessment_type = 'TMA'
  AND is_banked = 0
  AND days_late IS NOT NULL
GROUP BY days_late
ORDER BY days_late;"""

SQL["q6_iqr"] = """WITH x AS (
  SELECT days_late
  FROM {CLEAN}
  WHERE {VARIANT}
    AND assessment_type = 'TMA'
    AND is_banked = 0
    AND days_late IS NOT NULL
),
q AS (
  SELECT DISTINCT
    PERCENTILE_CONT(days_late, 0.25) OVER () AS q1,
    PERCENTILE_CONT(days_late, 0.75) OVER () AS q3
  FROM x
)
SELECT
  q1,
  q3,
  q3 - q1                 AS iqr,
  q1 - 1.5 * (q3 - q1)    AS lower_bound,
  q3 + 1.5 * (q3 - q1)    AS upper_bound,
  COUNTIF(x.days_late < q1 - 1.5 * (q3 - q1)
       OR x.days_late > q3 + 1.5 * (q3 - q1)) AS n_outliers,
  COUNT(*)                AS n_total
FROM x CROSS JOIN q
GROUP BY q1, q3;"""

SQL["q6b_banked"] = """SELECT
  is_banked,
  COUNT(*)                    AS n_rows,
  MIN(date_submitted)         AS min_day,
  MAX(date_submitted)         AS max_day,
  COUNTIF(date_submitted < 0) AS n_negative_day
FROM {CLEAN}
WHERE {VARIANT}
GROUP BY is_banked
ORDER BY is_banked;"""

SQL["q7_missing_structure"] = """SELECT
  final_result,
  COUNT(*)                                            AS n_rows,
  COUNTIF(score_raw = '?')                            AS n_question_mark,
  COUNTIF(score_raw IN ('999', '-1'))                 AS n_other_codes,
  ROUND(100 * COUNTIF(score_raw = '?') / COUNT(*), 3) AS pct_question_mark,
  ROUND(100 * COUNTIF(score_raw IN ('999', '-1')) / COUNT(*), 3) AS pct_other_codes
FROM {CLEAN}
GROUP BY final_result
ORDER BY pct_question_mark DESC;"""

SQL["q8_check"] = """SELECT
  id_assessment,
  assessment_type,
  COUNT(*)               AS n_rows,
  COUNT(score)           AS n_scored,
  ROUND(AVG(score), 2)   AS mean_score,
  COUNTIF(days_late > 0) AS n_late
FROM {CLEAN}
WHERE {VARIANT}
GROUP BY id_assessment, assessment_type
ORDER BY id_assessment;"""

SQL["q9_create_analytic"] = """CREATE OR REPLACE VIEW {ANALYTIC} AS
SELECT
  code_module,
  code_presentation,
  id_student,
  ANY_VALUE(final_result)                     AS final_result,
  COUNT(*)                                    AS n_submissions,
  COUNT(score)                                AS n_scored,
  ROUND(AVG(score), 2)                        AS mean_score,
  ROUND(SAFE_DIVIDE(
    COUNTIF(assessment_type = 'TMA' AND days_late > 0),
    COUNTIF(assessment_type = 'TMA' AND days_late IS NOT NULL)), 3) AS share_late_tma,
  COUNTIF(score_flag != 'ok')                 AS n_flagged
FROM {CLEAN}
GROUP BY code_module, code_presentation, id_student;"""

SQL["q9b_analytic_by_result"] = """SELECT
  final_result,
  COUNT(*)                       AS students,
  ROUND(AVG(mean_score), 2)      AS avg_mean_score,
  ROUND(AVG(share_late_tma), 3)  AS avg_share_late_tma,
  COUNTIF(n_flagged > 0)         AS students_with_flags
FROM {ANALYTIC}
WHERE {VARIANT}
GROUP BY final_result
ORDER BY avg_mean_score DESC;"""

print(f"Запросов: {len(SQL)}. Пример — первый запрос вашего варианта:\n")
print(sql("q0_control"))

### 9.4. Загрузка `practice02_raw.csv` в BigQuery: все 12 полей — STRING

Raw-слой хранит выгрузку «как есть», поэтому **все поля должны быть `STRING`** — это контракт данных из лекции 2:
типы приводятся позже, в clean-слое, через `SAFE_CAST`. Автоопределение схемы (**Auto detect**) могло бы решить,
что `score` — число, и упасть на значении `'?'`.

Загрузить можно одним из двух способов (результат одинаковый):

**Способ 1 — из Colab (рекомендуется).** Выполните ячейку 9.4б с `LOAD_FROM_COLAB = True`. Схема STRING и пропуск
строки заголовка заданы в коде — ошибиться негде.

**Способ 2 — вручную в консоли BigQuery** (для знакомства с интерфейсом):

1. Выполните ячейку 9.4а: она напечатает схему **одной строкой**.
2. Датасет вашей группы → ⋮ → **Create table**.
3. **Source:** `Create table from` = **Upload**, файл `practice02_raw.csv`, `File format` = **CSV**.
4. **Destination:** `Table` = `p02_raw`, `Table type` = Native table.
5. **Schema:** снимите **Auto detect**, включите **Edit as text** — и больше эти переключатели не трогайте.
6. Щёлкните **внутри поля редактора**, справа от цифры 1, и вставьте строку из ячейки 9.4а (Ctrl+V).
   Строка должна появиться в поле. Пустое поле — ошибка *Empty schema specified*.
7. **Advanced options → Header rows to skip = 1**. Без этого строка заголовка загрузится как данные,
   и ошибки не будет: её покажет только запрос 1 (`header_as_data = 1`).
8. **Create table**. Затем в ячейке 9.4б поставьте `LOAD_FROM_COLAB = False`.

Проверка: `p02_raw → Schema` — 12 полей типа `STRING`; **Preview** — первая строка содержит данные, а не заголовки.

In [ ]:
# 9.4а. Схема для консоли BigQuery (способ 2): строится из заголовка файла
import json

cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
schema = [{"name": c, "type": "STRING", "mode": "NULLABLE"} for c in cols]
Path("p02_raw_schema.json").write_text(json.dumps(schema, ensure_ascii=False))
print(f"Столбцов: {len(cols)}. Скопируйте строку ниже целиком в поле Edit as text:\n")
print(json.dumps(schema))

In [ ]:
# 9.4б. Загрузка из Colab (способ 1). Повторный запуск безопасен: таблица перезаписывается.
LOAD_FROM_COLAB = True        # False — если таблица уже загружена вручную через консоль

if ENGINE == "bigquery" and LOAD_FROM_COLAB:
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,                                   # строка заголовка — не данные
        schema=[bigquery.SchemaField(c, "STRING") for c in cols],
        autodetect=False,
        write_disposition="WRITE_TRUNCATE",
    )
    with open(DATA_PATH, "rb") as f:
        client.load_table_from_file(f, f"{PROJECT_ID}.{DATASET}.{TABLES['RAW']}", job_config=job_config).result()
    t = client.get_table(f"{PROJECT_ID}.{DATASET}.{TABLES['RAW']}")
    print(f"Загружено: {t.num_rows:,} строк; типы полей: {sorted({f.field_type for f in t.schema})}")
elif ENGINE == "bigquery":
    print("Используется таблица, загруженная вручную через консоль.")
else:
    print(f"Режим duckdb: таблица {TABLES['RAW']} создана локально из {DATA_PATH} (все поля — текст).")

### 9.5. Серия запросов

Выполняйте ячейки по порядку. Каждая печатает SQL — его можно скопировать в консоль BigQuery и получить тот же результат.

#### Запрос 1. Контроль загрузки — `q0_control`

Считаем строки, события, студентов и курсы во **всей** таблице и проверяем, не попала ли строка заголовка в данные.
Это «предполётная проверка»: если числа не совпали — дальше идти нельзя.

In [ ]:
q0 = run("q0_control")
display(q0)
ok = (int(q0.rows_count[0]) == len(raw_all)) and (int(q0.header_as_data[0]) == 0)
print("Таблица загружена верно" if ok else
      "ПРОВЕРЬТЕ ЗАГРУЗКУ: число строк не совпало с pandas или заголовок попал в данные (Header rows to skip = 1)")

**Контрольные значения (одинаковы у всех):** 176 172 строки · 174 781 событие · 23 369 студентов · 22 презентации ·
`header_as_data = 0`. Если строк 176 173 и `header_as_data = 1` — при ручной загрузке не указан *Header rows to skip = 1*.

#### Запрос 2. Сколько данных в каждом курсе — `q0b_presentations`

Группируем строки по курсу и потоку (`GROUP BY`). Оранжевый столбец — ваш вариант.

In [ ]:
q0b = run("q0b_presentations")
my = f"{MODULE} {PRESENTATION}"
bars(q0b["presentation"], q0b["n_rows"], "Строк в каждом курсе и потоке (оранжевый — ваш вариант)",
     "строк", colors=[ORANGE if p == my else GREY for p in q0b["presentation"]], horizontal=True, figsize=(9, 6.5))
print(f"Ваш вариант {my}: {int(q0b.loc[q0b.presentation == my, 'n_rows'].iloc[0]):,} строк (в pandas: {len(raw):,})")

> **Что вы видите?** Большой или маленький у вас поток по сравнению с другими? Почему доли (C, V, U, S) можно сравнивать между вариантами, а абсолютные числа — нет?
>
> **Ваш ответ:** ...

#### Запрос 3. Что записано в поле `score` — `q1_profile` (TODO-4)

Раскладываем значения балла на виды: нормальный балл 0–100, код `'?'`, другие коды «нет данных», пустое значение,
число вне шкалы. Запрос работает с **текстом** (`STRING`), поэтому видит коды «как есть».

**TODO-4.** В строке `WHEN score IN (___)` впишите через запятую **остальные служебные коды** «нет данных»
(те же, что вы добавили в `MISSING_CODES` на шаге 3, кроме `'?'`). Коды — текст, поэтому каждый в кавычках:
`'…', '…'`.

In [ ]:
SQL["q1_profile"] = """SELECT
  CASE
    WHEN score IS NULL                          THEN '4. пусто'
    WHEN score = '?'                            THEN '2. код ?'
    WHEN score IN (___)                         THEN '3. код 999 / -1'   -- TODO-4
    WHEN SAFE_CAST(score AS INT64) BETWEEN 0 AND 100 THEN '1. балл 0–100'
    ELSE '5. вне шкалы'
  END      AS score_kind,
  COUNT(*) AS n_rows
FROM {RAW}
WHERE {VARIANT}
GROUP BY score_kind
ORDER BY score_kind;"""

q1 = run("q1_profile")
display(q1)
bars(q1["score_kind"], q1["n_rows"], f"Виды значений score в raw-слое, {MODULE} {PRESENTATION}",
     "строк", colors=[BLUE] + [ORANGE] * (len(q1) - 1))

# Автопроверка TODO-4: сравниваем с правилами pandas (шаг 3)
codes_py = int(raw["score"].isin(MISSING_CODES - {"?"}).sum())
num_py = pd.to_numeric(raw["score"].where(~raw["score"].isin(MISSING_CODES)), errors="coerce")
out_py = int((num_py.notna() & ~num_py.between(0, 100)).sum())
got = q1.set_index("score_kind")["n_rows"]
if int(got.get("3. код 999 / -1", 0)) == codes_py and int(got.get("5. вне шкалы", 0)) == out_py:
    print(f"TODO-4 верно: кодов 999/-1 — {codes_py}, вне шкалы — {out_py} (как в pandas)")
else:
    print(f"TODO-4: не совпало с pandas (ожидалось кодов {codes_py}, вне шкалы {out_py}). Проверьте коды в IN (...)")

> **Что вы видите?** Сколько у вас кодов пропуска и сколько значений вне шкалы? Почему `999` и `-1` — это коды, а `101–120` — ошибки ввода?
>
> **Ваш ответ:** ...

#### Запрос 4. Как записан тип задания — `q1b_types`

Считаем все варианты написания `assessment_type`. Ожидаются только `TMA`, `CMA`, `Exam`; оранжевые — нестандартные.

In [ ]:
q1b = run("q1b_types")
display(q1b)
std = {"TMA", "CMA", "Exam"}
bars(q1b["assessment_type"], q1b["n_rows"], "Написания типа задания (оранжевый — нестандартные)", "строк",
     colors=[BLUE if t in std else ORANGE for t in q1b["assessment_type"]])
print("Нестандартных написаний:", int(q1b.loc[~q1b.assessment_type.isin(std), "n_rows"].sum()))

> **Что вы видите?** Сколько строк с нестандартным написанием? Какой функцией SQL их можно привести к словарю (подсказка — запрос 7.4)?
>
> **Ваш ответ:** ...

#### Запрос 5. Дубли — `q2_duplicates`, `q2b_dup_examples`

Три числа: все строки → строки без точных повторов (`SELECT DISTINCT *`) → уникальные пары «задание — студент».
Разница между первым и вторым — **точные** дубли, между вторым и третьим — **логические**.

In [ ]:
q2 = run("q2_duplicates")
r = q2.iloc[0]
exact, logical = int(r.n_rows - r.n_distinct_rows), int(r.n_distinct_rows - r.n_keys)
bars(["точные дубли\n(строка повторена целиком)", "логические дубли\n(тот же ключ, новый event_id)"], [exact, logical],
     f"Дубли в {MODULE} {PRESENTATION}: из {int(r.n_rows):,} строк останется {int(r.n_keys):,}".replace(",", " "),
     "строк", colors=[ORANGE, BLUE], figsize=(8, 3.4))
print(f"Точные дубли: {exact} (pandas: {len(raw) - len(step1)}); логические: {logical} (pandas: {len(step2) - len(step3)})")

> **Что вы видите?** Сколько точных и логических дублей? Почему `SELECT DISTINCT` не находит логические дубли?
>
> **Ваш ответ:** ...

**Примеры логических дублей.** У копий разные `event_id`, а балл может отличаться — нужно правило, какую запись оставить.

In [ ]:
run("q2b_dup_examples")

> **Что вы видите?** Отличаются ли баллы у копий? Какую запись мы оставляем в clean-слое и почему это правило нужно записать в паспорт?
>
> **Ваш ответ:** ...

#### Запрос 6. Показатели качества raw-слоя — `q3_raw_quality`

Четыре показателя качества из лекции 2 и общий индекс: полнота **C**, допустимость **V**, уникальность **U**,
согласованность **S**, $DQ = 0{,}35C + 0{,}30V + 0{,}20U + 0{,}15S$.

In [ ]:
q3 = run("q3_raw_quality")
display(q3)
m = ["C", "V", "U", "S", "DQ"]
bars(m, q3.loc[0, m], f"Качество raw-слоя, {MODULE} {PRESENTATION}", "доля", colors=[BLUE] * 4 + [ORANGE], fmt="{:.4f}")
print("DQ raw в pandas (шаг 4):", q_raw["DQ"])

> **Что вы видите?** Совпал ли DQ с pandas? Все показатели выше 0,98 — значит ли это, что данные можно использовать «как есть»?
>
> **Ваш ответ:** ...

#### Запрос 7. Clean-слой по шагам: четыре представления

Clean-слой строится **серией представлений** (`VIEW`): каждое — один шаг очистки поверх предыдущего. Представление
не копирует данные: raw-слой остаётся неизменным, а очистка пересчитывается при каждом запросе.

```
p02_raw → p02_s1_distinct → p02_s2_typed → p02_s3_dedup → p02_clean
          (точные дубли)    (типы, коды)   (логич. дубли)  (категории, флаги, days_late)
```

Представления создаются **один раз на всю таблицу** (без фильтра варианта) — фильтр применяется при запросах к ним.
Ячейка печатает «Представление создано: …» — таблицы результата у `CREATE` нет.

**7.1. Без точных дублей** (аналог `drop_duplicates()` в шаге 5).

In [ ]:
run("q4a_s1_distinct")

**7.2. Типы и служебные коды.** `SAFE_CAST` возвращает `NULL` вместо ошибки — аналог `errors="coerce"` в pandas.

In [ ]:
run("q4b_s2_typed")

**7.3. Без логических дублей (TODO-5).** Для каждой пары «задание — студент» нужно оставить **первую доставку**.
`ROW_NUMBER()` нумерует строки внутри группы, `QUALIFY … = 1` оставляет первую.

**TODO-5.** Допишите строку `QUALIFY`:
* после `PARTITION BY` — **ключ** (по каким полям ищем повторы; тот же `KEY`, что в pandas);
* после `ORDER BY` — поле, по которому определяется «первая» доставка (в pandas: `sort_values(...)`).

In [ ]:
SQL["q4c_s3_dedup"] = """CREATE OR REPLACE VIEW {S3} AS
SELECT *                   -- шаг 3: из повторов одного ключа оставить первую доставку
FROM {S2}
WHERE TRUE
QUALIFY ROW_NUMBER() OVER (PARTITION BY ___ ORDER BY ___) = 1;   -- TODO-5"""

run("q4c_s3_dedup")

**7.4. Итоговое clean-представление (TODO-6).** Нормализация типа задания, балл вне шкалы → `NULL` с флагом,
флаг согласованности и признак опоздания `days_late`.

**TODO-6.** Впишите вместо `___` выражение для `days_late` — на сколько дней сдача отличается от срока
(то же, что в шаге 5 pandas: `clean["days_late"] = …`). Имена полей — `date_submitted` и `assessment_deadline`.

In [ ]:
SQL["q4_create_clean"] = """CREATE OR REPLACE VIEW {CLEAN} AS
SELECT                     -- шаги 4–6: нормализация категорий, правила для балла, признаки
  event_id,
  code_module,
  code_presentation,
  id_assessment,
  CASE UPPER(TRIM(assessment_type_raw))
    WHEN 'TMA'  THEN 'TMA'
    WHEN 'CMA'  THEN 'CMA'
    WHEN 'EXAM' THEN 'Exam'
  END                                                    AS assessment_type,
  assessment_deadline,
  weight,
  id_student,
  date_submitted,
  is_banked,
  score_raw,
  IF(score_int BETWEEN 0 AND 100, score_int, NULL)       AS score,
  CASE
    WHEN score_raw IS NULL OR score_raw IN ('?', '999', '-1') THEN 'missing_code'
    WHEN score_int IS NULL OR score_int NOT BETWEEN 0 AND 100 THEN 'out_of_range'
    ELSE 'ok'
  END                                                    AS score_flag,
  final_result,
  ___                                                    AS days_late,       -- TODO-6
  (final_result = 'Withdrawn'
   AND UPPER(TRIM(assessment_type_raw)) = 'EXAM')        AS withdrawn_exam
FROM {S3};"""

run("q4_create_clean")

**7.5. Журнал строк (lineage) — `q4e_lineage`.** Сколько строк вашего варианта осталось после каждого шага.
Автопроверка сравнивает журнал с журналом pandas (шаг 5) — так проверяется TODO-5.

In [ ]:
q4e = run("q4e_lineage")
display(q4e)
bars(q4e["step"], q4e["n_rows"], f"Журнал строк clean-слоя, {MODULE} {PRESENTATION}", "строк",
     colors=[GREY, BLUE, BLUE, GREEN])

expected = [len(raw), len(step1), len(step3), len(clean)]
got = [int(v) for v in q4e["n_rows"]]
print("pandas  :", " → ".join(map(str, expected)))
print("BigQuery:", " → ".join(map(str, got)))
print("TODO-5 верно: журналы совпали" if got == expected else
      "TODO-5: журналы не совпали — проверьте PARTITION BY (ключ) и ORDER BY в запросе 7.3")

> **Что вы видите?** На каком шаге уходит больше всего строк? Почему последний шаг (clean) строк не удаляет?
>
> **Ваш ответ:** ...

#### Запрос 8. Качество после очистки — `q5_clean_quality`

Те же показатели для clean-слоя и сравнение с raw на одном графике.

In [ ]:
q5 = run("q5_clean_quality")
display(q5)
x = np.arange(len(m)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 3.6))
b1 = ax.bar(x - w / 2, q3.loc[0, m].astype(float), w, color=GREY, label="raw")
b2 = ax.bar(x + w / 2, q5.loc[0, m].astype(float), w, color=GREEN, label="clean")
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x, m); ax.set_ylim(0.97, 1.003); ax.set_ylabel("доля")
ax.set_title("Качество до и после очистки (ось начинается с 0,97 — чтобы различия были видны)")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1)); plt.tight_layout(); plt.show()
print("DQ clean в pandas (шаг 6):", q_clean["DQ"])

> **Что вы видите?** Какие показатели выросли, а какой снизился? Почему снижение полноты C после очистки — это честнее, а не хуже? Почему на графике ось обрезана и чем это опасно?
>
> **Ваш ответ:** ...

#### Запрос 9. Что помечено флагами — `q5b_flags`

Какие исходные значения балла получили флаг `missing_code` или `out_of_range`. Исходное значение хранится в
`score_raw` — ничего не потеряно.

In [ ]:
q5b = run("q5b_flags")
display(q5b)
lab = q5b["score_flag"] + ": " + q5b["score_raw"].astype(str)
bars(lab, q5b["n_rows"], "Значения балла, помеченные флагами", "строк", horizontal=True,
     colors=[ORANGE if f == "missing_code" else BLUE for f in q5b["score_flag"]], figsize=(9, 0.35 * len(q5b) + 1.2))

> **Что вы видите?** Какой код пропуска встречается чаще всего? Чем значения `out_of_range` отличаются от кодов — почему это опечатки?
>
> **Ваш ответ:** ...

#### Запрос 10. Опоздания и выбросы — `q6a_days_late`, `q6_iqr`, `q6b_banked`

Распределение отклонения от срока `days_late` для TMA без перезачтённых работ и границы выбросов по правилу IQR.
Автопроверка сравнивает результат с pandas (шаг 7) — так проверяется TODO-6.

In [ ]:
q6a = run("q6a_days_late")
q6 = run("q6_iqr")
display(q6)
lo, hi = float(q6.loc[0, "lower_bound"]), float(q6.loc[0, "upper_bound"])
view = q6a[(q6a.days_late >= -30) & (q6a.days_late <= 30)]
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(view["days_late"], view["n_rows"], color=[BLUE if lo <= d <= hi else ORANGE for d in view["days_late"]], width=0.9)
ax.axvline(lo, color="#e34948", ls="--", lw=1.2); ax.axvline(hi, color="#e34948", ls="--", lw=1.2, label=f"границы IQR [{lo:g}; {hi:g}]")
ax.set(title="Отклонение сдачи TMA от срока, дни (оранжевый — выбросы по IQR; показан диапазон ±30)",
       xlabel="days_late (< 0 — раньше срока, > 0 — позже)", ylabel="сдач")
ax.legend(frameon=False); plt.tight_layout(); plt.show()
print(f"Выбросов: {int(q6.loc[0, 'n_outliers']):,} из {int(q6.loc[0, 'n_total']):,} "
      f"({q6.loc[0, 'n_outliers'] / q6.loc[0, 'n_total'] * 100:.1f} %)")

# Автопроверка TODO-6: сравниваем с pandas
tma_py = clean[(clean["assessment_type"] == "TMA") & (clean["is_banked"] == 0)]["days_late"].dropna()
py_q1, py_q3 = tma_py.quantile([0.25, 0.75])
same = (int(q6.loc[0, "n_total"]) == len(tma_py) and float(q6.loc[0, "q1"]) == py_q1 and float(q6.loc[0, "q3"]) == py_q3)
print("TODO-6 верно: Q1, Q3 и число сдач совпали с pandas" if same else
      f"TODO-6: не совпало с pandas (Q1 = {py_q1:g}, Q3 = {py_q3:g}, сдач {len(tma_py)}). Проверьте выражение days_late")

> **Что вы видите?** Какая доля сдач — «выбросы» по IQR? Можно ли их удалить? Почему правило IQR плохо работает для несимметричного распределения?
>
> **Ваш ответ:** ...

**Перезачтённые работы.** Почему они исключены из анализа опозданий.

In [ ]:
q6b = run("q6b_banked")
display(q6b)

> **Что вы видите?** Какие даты сдачи у перезачтённых работ (`is_banked = 1`)? Почему их нельзя использовать для оценки своевременности?
>
> **Ваш ответ:** ...

#### Запрос 11. У кого чаще пропуски балла — `q7_missing_structure`

Доля кода `'?'` по итогу курса — по **всему** файлу (без фильтра варианта), чтобы чисел хватило для сравнения.

In [ ]:
q7 = run("q7_missing_structure")
display(q7)
bars(q7["final_result"], q7["pct_question_mark"], "Доля кода «?» в балле по итогу курса, % (весь файл)", "%",
     colors=[ORANGE if r == "Withdrawn" else BLUE for r in q7["final_result"]], fmt="{:.3f}")

> **Что вы видите?** У какой группы доля `'?'` наибольшая? Похоже ли это на MCAR? Что случится с выборкой, если удалить строки с пропусками? Совпало ли с первой таблицей шага 8?
>
> **Ваш ответ:** ...

### 9.6. Сводка: BigQuery против Colab

Перенесите значения из ответов BigQuery и сравните с шагами 2–7 в Colab.

| Показатель | Colab (pandas) | BigQuery | Совпало? |
|---|---|---|---|
| Запрос 1: строк / событий / презентаций | 176172 / 174781 / 22 |  |  |
| Запрос 3: кодов `?` / кодов 999 и -1 / вне шкалы |  |  |  |
| Запрос 5: точных / логических дублей |  |  |  |
| Запрос 6: DQ raw |  |  |  |
| Запрос 7.5: журнал строк raw → clean |  |  |  |
| Запрос 8: строк clean / DQ clean |  |  |  |
| Запрос 10: Q1 / Q3 / число кандидатов в выбросы |  |  |  |

### 9.7. Собственный SQL-запрос (TODO-7)

Запрос `q7_missing_structure` (запрос 11) считает доли кодов по **всему** файлу. Напишите свой запрос, который
считает то же самое **только для вашего варианта**, выполните его и постройте график.

**Как сделать:**
1. Скопируйте текст `q7_missing_structure` из вывода запроса 11 **или** из ячейки 9.3 (вариант с метками `{CLEAN}` удобнее).
2. Добавьте фильтр варианта: строку `WHERE {VARIANT}` (между `FROM` и `GROUP BY`).
3. Выполните ячейку. Автопроверка сравнит результат со второй таблицей шага 8 (`missing_table(clean)`).
4. Постройте график функцией `bars(подписи, значения, "заголовок", "подпись оси")` — например, доля кодов
   (`pct_question_mark + pct_other_codes`) по итогу курса.

In [ ]:
# TODO-7. Собственный запрос: доли кодов пропуска по итогу курса — только для вашего варианта
MY_SQL = """
-- вставьте сюда свой SQL (метки {CLEAN} и {VARIANT} подставятся автоматически)
"""

my = run_custom(MY_SQL)
display(my)

# Автопроверка: сравнение с pandas (шаг 8)
py = missing_table(clean)
cols = ["n_rows", "n_question_mark", "n_other_codes"]
same = (my.set_index("final_result")[cols].astype(int).sort_index()
          .equals(py[cols].astype(int).sort_index()))
print("Совпадает с missing_table(clean) из шага 8:", same)

# TODO-7 (продолжение). Постройте график своего результата функцией bars(...)

> **Что вы видите?** Совпал ли результат с Colab? Похожа ли картина в вашем срезе на картину по всему файлу (запрос 11)? Почему выводы по одному потоку нужно делать осторожно?
>
> **Ваш ответ:** ...

## Шаг 10. Сверка Python и BigQuery (запрос 12)

Смысл шага — не «SQL правильнее Python», а **одинаковый результат в двух средах**. Сверочный запрос `q8_check`
считает по каждому заданию число строк, баллов, средний балл и число опозданий; те же величины считаются в pandas
по таблице `clean` (шаг 5).

1. Первая ячейка считает таблицу `chk_py` в pandas.
2. Вторая выполняет `q8_check` в BigQuery, сравнивает и рисует график: **столбцы — SQL, точки — pandas**.
3. Третья нужна, только если вы выполняли запросы вручную в консоли: `q8_check` → **Save results → CSV (local file)**
   → переименуйте в `bq_p02_check.csv` и загрузите, когда Colab попросит.

In [ ]:
chk_py = (clean.groupby(["id_assessment", "assessment_type"])
          .agg(n_rows=("event_id", "size"),
               n_scored=("score", "count"),
               mean_score=("score", "mean"),
               n_late=("days_late", lambda s: int((s > 0).sum())))
          .reset_index())
chk_py["mean_score"] = chk_py["mean_score"].round(2)


def compare_with(other, label):
    cols = ["id_assessment", "assessment_type", "n_rows", "n_scored", "mean_score", "n_late"]
    a = chk_py[cols].sort_values("id_assessment").reset_index(drop=True)
    b = other[cols].sort_values("id_assessment").reset_index(drop=True)
    b["id_assessment"] = b["id_assessment"].astype("int64")
    try:
        pd.testing.assert_frame_equal(a, b, check_dtype=False, check_exact=False, atol=0.011, rtol=0)
        print(f"Сверка с {label} пройдена: {len(a)} заданий, все показатели совпали.")
    except AssertionError as e:
        print(f"Расхождение с {label}:\n", str(e)[:800])


chk_py

In [ ]:
# Запрос 12. Сверочный запрос q8_check: результат BigQuery против pandas
q8 = run("q8_check")
compare_with(q8, "BigQuery" if ENGINE == "bigquery" else "SQL-эталоном (DuckDB)")

fig, ax = plt.subplots(figsize=(10, 3.6))
lab = q8["id_assessment"].astype(int).astype(str) + "\n" + q8["assessment_type"]
ax.bar(lab, q8["mean_score"].astype(float), color=BLUE, label="SQL (q8_check)")
ax.plot(lab, chk_py.sort_values("id_assessment")["mean_score"].values, "o", color=ORANGE, label="pandas (clean)")
ax.set(title="Средний балл по заданиям: столбцы — SQL, точки — pandas (должны совпасть)", ylabel="средний балл", ylim=(0, 100))
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1)); plt.tight_layout(); plt.show()

In [ ]:
# Только если запросы выполнялись вручную в консоли BigQuery (или Colab не подключился к BigQuery):
# q8_check → Save results → CSV (local file) → bq_p02_check.csv → загрузить сюда
USE_CSV = False

if USE_CSV:
    BQ_FILE = Path("bq_p02_check.csv")
    if not BQ_FILE.exists():
        from google.colab import files
        print("Загрузите bq_p02_check.csv — результат запроса q8_check из консоли BigQuery.")
        up = files.upload()
        BQ_FILE = Path(next(iter(up)))
    compare_with(pd.read_csv(BQ_FILE), "файлом из консоли BigQuery")
else:
    print("Файл из консоли не нужен: сверка выполнена в предыдущей ячейке.")

> **Что вы видите?** Сверка пройдена? Легли ли точки pandas на вершины столбцов? Если нет — назовите причину
> (чужой вариант, таблица загружена с Auto detect или без пропуска заголовка, ячейки не перезапущены после правки TODO).
>
> **Ваш ответ:** ...

## Шаг 11. ★ Аналитический слой (по желанию, не оценивается)

Ещё одно представление `p02_analytic` — **одна строка на студента**: число сдач, средний балл, доля поздних TMA,
число флагов. Запрос `q9b_analytic_by_result` сравнивает группы студентов по итогу курса. Оно понадобится в ЛР № 2.

In [ ]:
run("q9_create_analytic")
q9 = run("q9b_analytic_by_result")
display(q9)
order = ["Distinction", "Pass", "Fail", "Withdrawn"]
q9 = q9.set_index("final_result").reindex([o for o in order if o in set(q9.final_result)]).reset_index()
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
axs[0].bar(q9["final_result"], q9["avg_mean_score"].astype(float), color=BLUE)
axs[0].set(title="Средний балл студента по итогу курса", ylabel="балл", ylim=(0, 100))
axs[1].bar(q9["final_result"], q9["avg_share_late_tma"].astype(float), color=ORANGE)
axs[1].set(title="Средняя доля поздних TMA по итогу курса", ylabel="доля")
for ax, col, f in ((axs[0], "avg_mean_score", "{:.1f}"), (axs[1], "avg_share_late_tma", "{:.3f}")):
    for i, v in enumerate(q9[col].astype(float)):
        ax.text(i, v, f.format(v), ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

> **Что вы видите?** Как связаны средний балл и доля поздних сдач с итогом курса?
>
> **Подумайте:** можно ли использовать `mean_score` из этого представления, чтобы **в начале курса** прогнозировать
> итог? Что такое утечка данных (лекция 2, раздел 13)?
>
> **Ваш ответ:** ...

## Шаг 12. Паспорт набора данных и вывод

### Паспорт (заполните фактическими числами своего варианта)

```text
Набор:           <проект>.<датасет>.p02_clean (датасет: __________), срез ____ ____
Источник:        OULAD (UCI, CC BY 4.0) + учебная выгрузка practice02_raw.csv
Raw-слой:        <датасет>.p02_raw, все поля STRING; загрузка: из Colab / вручную в консоли
Строк:           raw ____ → −____ точных дублей → ____ → −____ логических дублей → clean ____
Ключ:            (id_assessment, id_student); при логических дублях оставлена ____________
score:           коды пропуска {________} → NULL (missing_code: ____); вне шкалы → NULL (out_of_range: ____)
Категории:       ____ нестандартных написаний приведены к {TMA, CMA, Exam}
Качество:        DQ ____ → ____; C ____ → ____
Согласованность: записей Withdrawn + Exam — ____
Ограничения:     ______________________________________________
Clean-слой:      4 представления p02_s1_distinct → p02_s2_typed → p02_s3_dedup → p02_clean
Сверка:          q8_check ↔ pandas: совпало / не совпало
```

### Педагогический вывод (5–7 предложений)

Обязательно ответьте:

1. Насколько исходная выгрузка была пригодна для анализа «как есть»?
2. Как очистка изменила средний балл и число сдач? Почему это важно для выводов об успешности курса?
3. Почему строки с пропусками нельзя просто удалить?
4. Какие ограничения данных нужно сообщить педагогу или администрации?

**Запрещённые формулировки:** «отчисленные не сдают работы», «модель доказала», «удалили плохие данные».

> **Ваш вывод:**
> ...

---
## Чек-лист перед сдачей

- [ ] Заполнены ФИО, группа, номер варианта; `VARIANT` указан.
- [ ] Контрольные значения файла: 176 172 строки, 174 781 `event_id`, 22 презентации.
- [ ] Выполнено профилирование, записаны выводы.
- [ ] `MISSING_CODES` и `TYPE_MAP` заданы — «Правила заданы верно».
- [ ] Конвейер выполнен — «Все автопроверки пройдены».
- [ ] Таблица качества «до / после» и ответ, почему изменилась C.
- [ ] Ответы на вопросы шагов 7 и 8.
- [ ] TODO-3: указаны `PROJECT_ID` и `DATASET` своей группы.
- [ ] В BigQuery создана таблица `p02_raw`, все 12 полей `STRING`; запрос 1 — «Таблица загружена верно».
- [ ] TODO-4, TODO-5, TODO-6: пропуски `___` заполнены, все три автопроверки — «верно».
- [ ] Созданы 4 представления clean-слоя (запросы 7.1–7.4).
- [ ] Графики запросов 2–11 построены, под каждым заполнено поле «Что вы видите?».
- [ ] Заполнена сводка 9.6.
- [ ] TODO-7: собственный запрос 9.7 выполнен, автопроверка `True`, построен график.
- [ ] Шаг 10: сверка `q8_check` с pandas пройдена, график построен.
- [ ] Заполнен паспорт данных, написан вывод без запрещённых формулировок.

### Что сдавать

Сохраните блокнот **с результатами выполнения** (таблицы и графики должны быть видны) как `Фамилия_Группа_ПР02.ipynb`:
**File / Файл → Download / Скачать → Download .ipynb**. Скриншоты консоли BigQuery не нужны, если запросы выполнены из Colab.